# Talent Intelligence Platform
### Investment Recruiting · Data Science Case Study

**A structured talent search workflow designed for Business Development teams**

The product is built around a simple principle: **make resume screening faster without making the underlying decisions less trustworthy.**

### Product proposition

**1. Reliable Matching**  
**Required criteria determine eligibility. Ranking signals determine priority.**  
Hard constraints are applied first; softer, job-relevant signals rank the candidates who remain. Near matches are surfaced separately rather than mixed into the qualified pool.

**2. Trusted Candidate Data**  
**Structured resume data with evidence, validation, and review flags.**  
The system extracts the information a screener actually needs, verifies extracted evidence against the source text, and flags issues that may require human review. The goal is not to automate judgment; it is to make the information behind that judgment more reliable and easier to review.

**3. Built for BD Workflows**  
**From candidate search to outreach and talent-pool insight.**  
The interface organizes candidate information around screening decisions, then extends the workflow into outreach, pool-level analysis, and sourcing opportunities.

> **AI is underneath the product, not the product itself.**  
> The product value comes from reliable data, explicit matching logic, and a workflow that helps a human make a better decision faster.

---

### What this case study demonstrates

- Resume parsing into a structured, validated candidate record
- Evidence-backed data quality checks
- Job-specific eligibility and ranking logic
- Separate treatment of qualified candidates and near matches
- Evaluation against blind human screening decisions
- A Streamlit interface designed for BD users
- A path from a 10-resume prototype to a production-scale talent data system


## The Business Problem

Millennium's Business Development team may need to source junior investment talent across different regions, investment approaches, sectors, and experience levels.

The challenge is not simply finding keywords in resumes. A useful screening system needs to answer three questions reliably:

1. **Is this candidate eligible for the role?**
2. **If eligible, what makes the candidate more relevant than another qualified candidate?**
3. **What information should a recruiter verify before acting on the result?**

The original case study asked for a searchable platform that could parse resumes, structure the resulting data, support multi-criteria search, visualize the candidate pool, and scale beyond the initial sample. fileciteturn7file0L17-L44

### Product approach

I separated the workflow into four layers:

**Resume → Structured data → Matching → BD action**

This separation matters. A parsing error should not silently become a matching decision, and a matching score should not hide the evidence behind it.

The current prototype uses 10 sample resumes and precomputed candidate data for the public application. The architecture is designed so that parsing, matching, evaluation, and serving can evolve independently.


## 1 · System Architecture

The platform separates **data extraction, domain knowledge, matching, evaluation, and user experience**.

![Pipeline](https://raw.githubusercontent.com/yc4379-commits/m-case-study/main/docs/architecture.png)

At a high level:

1. **Read** the source document and recover usable text.
2. **Parse** the resume into a typed candidate record.
3. **Validate** the extracted information and preserve source evidence.
4. **Enrich** the record with curated domain knowledge such as firm lineage and credential mappings.
5. **Match** candidates to a requisition using required criteria first and ranking signals second.
6. **Evaluate** the decisions against blind human screening.
7. **Serve** the structured data through a workflow designed for BD users.

The key design choice is that the LLM is responsible for **interpreting unstructured text**, while deterministic code is responsible for **validation, normalization, filtering, and scoring rules**.


Before matching begins, the platform creates an orientation view of the **entire parsed candidate pool**.

This is useful for two reasons:

- It gives the BD user a quick view of what the current talent pool actually contains.
- It makes data quality visible before the user starts relying on search results.

The product therefore treats the candidate record as a data asset, not just an intermediate LLM output.


In [1]:
import json, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

candidates = json.loads((ROOT / "data" / "candidates.json").read_text())
pd.DataFrame([{
    "candidate": c["display_name"],
    "region": c["region"],
    "approach": c["approach_family"],
    "yrs investing": c["years_investment_experience"],
    "sectors": ", ".join(c["sectors"]),
    "firm": c["current_firm"],
    "parse confidence": f'{c["quality"]["score"]} ({c["quality"]["band"]})',
} for c in candidates])

,candidate,region,approach,yrs investing,sectors,firm,parse confidence
0,Chen Li (Alex),APAC,systematic_quant,2.2,"technology, healthcare",NaN,0.96 (high)
1,MARINA SILVA COSTA,Europe,fundamental,5.0,"healthcare, consumer, media_telecom",Vanguard Group,0.8 (high)
2,Marcus Chen-Rodriguez,US,fundamental,9.2,healthcare,Coatue Management,0.8 (high)
3,Michael Rodriguez,US,fundamental,8.0,"technology, media_telecom, credit",Fidelity Asset Management,0.9 (high)
4,Omar El-Hassan,Europe,systematic_quant,0.0,financials,BNP Paribas,0.83 (high)
5,Priya Nakamura,APAC,fundamental,12.7,healthcare,ICICI Securities,0.65 (medium)
6,Ryan Patel,US,fundamental,6.3,"consumer, technology, healthcare",Meridian Capital Partners,0.88 (high)
7,Vikram Shah,US,fundamental,7.5,"technology, media_telecom",Cinctive Capital Management,0.98 (high)
8,Viktor Sharat,APAC,fundamental,9.8,"healthcare, energy, industrials",NaN,0.46 (low)
9,Dr. Zara Al-Rashid,APAC,fundamental,10.6,healthcare,Meridian Research Partners,0.62 (medium)


## 2 · Building a Reliable Resume Data Layer

Resume parsing can fail before an LLM ever sees the information.

A resume may store content in multiple columns, tables, text boxes, or PDF encodings that do not extract cleanly. **A model cannot recover information it never received.**

The extraction layer therefore focuses on recovering the source document faithfully before interpretation begins.

Examples from the sample set include:

- DOCX resumes where important content sits inside text boxes
- Tables containing work history that must be read in document order
- PDFs with broken character mappings
- Two-column layouts that require layout-aware handling

The system records extraction diagnostics for each document rather than silently repairing or discarding content.

This creates a useful division of responsibility:

> **Recover the document first. Interpret it second. Verify the result third.**

That principle is especially important in recruiting because a missing employer, role, date, or credential can change the downstream match.


In [2]:
log = pd.read_csv(ROOT / "data" / "extraction_log.csv")
log

,source_file,file_type,char_count,table_count,table_share,textbox_count,textbox_chars,page_count,multi_column_detected,ligature_repairs,replacement_chars_remaining,warnings
0,Chen_Li_Alex.docx,docx,4270,0,0.000,0,0,0,False,0,0,NaN
1,MARINA_SILVA_COSTA.docx,docx,3499,0,0.000,0,0,0,False,0,0,NaN
2,Marcus_ChenRodriguez_Resume.docx,docx,3407,0,0.000,0,0,0,False,0,0,NaN
3,Michael_Rodriguez_CFA.docx,docx,3608,1,0.034,0,0,0,False,0,0,NaN
4,Omar_ElHassan_202405.pdf,pdf,1639,0,0.000,0,0,1,True,35,2,Multi-column layout detected - columns were se...
5,Priya_Nakamura_sellside_healthcare_RLTM.docx,docx,5188,1,0.049,0,0,0,False,0,0,NaN
6,RYAN_PATEL__Resume.pdf,pdf,4953,0,0.000,0,0,2,False,0,0,NaN
7,Vikram_Shah.docx,docx,3880,0,0.000,0,0,0,False,0,0,NaN
8,Viktor_Sharat.docx,docx,3100,2,0.824,3,40,0,False,0,0,3 floating text box(es) recovered and hoisted ...
9,Zara_AlRashid.docx,docx,3401,7,0.164,0,0,0,False,0,0,NaN


## 3 · Structured Resume Parsing

The objective is **not to extract as much information as possible**. It is to create a candidate record that is reliable enough to support search, ranking, and human review.

### Use the model for interpretation; use deterministic logic for verification.

The candidate schema in `src/schema.py` defines the expected structure of every parsed record. The schema is supplied to the model as the output contract, so downstream code consumes typed fields rather than model-generated prose.

Three controls protect the extraction step:

1. **Validation and corrective retry**  
   If the structured response fails validation, the specific errors are returned once for correction rather than triggering an unconstrained re-generation.

2. **Evidence verification**  
   Extracted claims include verbatim evidence from the resume. Each quote is checked against the recovered source text. If the evidence cannot be found, the claim is treated as unreliable rather than accepted because it sounds plausible.

3. **Content-hash caching**  
   Results are cached by source text, model, schema, and prompt. Re-running unchanged inputs is therefore inexpensive, while changes to the extraction logic automatically invalidate stale results.

The model is deliberately kept within a narrow role: **interpret the resume; do not invent missing facts.**

Fine-tuning is not used because the prototype does not have a labelled training set. Agents are unnecessary for a single structured extraction task. Semantic embeddings remain a replaceable option for larger corpora.

The current 10-resume build cost approximately **$0.80**, while the public application uses precomputed data and does not call a model.


In [3]:
from parse import SYSTEM_PROMPT
print(SYSTEM_PROMPT)

You extract structured data from investment-industry resumes for a hedge fund business development team.

Rules that matter more than completeness:

1. Transcribe, do not embellish. If the resume does not state something, return null. A null is useful; an invented value is a liability.

2. Evidence must be VERBATIM. Every `evidence` field must be a substring of the resume text, copied exactly. Never paraphrase, summarise or reconstruct a quote. If you cannot find supporting text, return an empty string and set confidence to "low".

3. Classify by substance, not vocabulary. What someone DID outranks what they called it. "Python" in a skills list is not evidence of systematic investing; a backtested factor model is.

4. Section headings lie. Some resumes file work history under "ACADEMIC PROFILE" or "KEY PROJECTS". An entry naming an employer, a role and a duration is a position regardless of the heading above it.

5. You are given the document text only, never its filename. If the resum

### A worked candidate record

**Ryan Patel** is used as the running example because his resume exercises several parts of the pipeline: employer normalization, platform lineage, tenure calculation, internship handling, and quantitative-claim extraction.

The important product principle is simple:

> **Every field shown in the application should be traceable to structured data and, where appropriate, source evidence.**

The application reads from the committed candidate dataset; it does not generate candidate facts on the fly.


In [4]:
ryan = next(c for c in candidates if c["display_name"] == "Ryan Patel")
e = ryan["extraction"]
pd.set_option("display.max_colwidth", 84)
clip = lambda t: t if len(t) <= 84 else t[:81] + "..."

def j(field, item):
    return {"field": field, "value": str(item["value"]),
            "conf": item["confidence"],
            "evidence (verbatim)": clip(item["evidence"])}

pd.DataFrame(
    [j("investment_approach", e["investment_approach"]),
     j("market_side", e["market_side"])]
    + [j("sector", s) for s in e["primary_sectors"]]
    + [j("asset_class", a) for a in e["asset_classes"]]
    + ([j("team_leadership", e["team_leadership"])]
       if e["team_leadership"]["value"] else [])
    + [{"field": f"stated_metric · {m['kind']}", "value": m["figure"],
        "conf": "", "evidence (verbatim)": clip(m["quote"])}
       for m in e["stated_metrics"] if m["kind"] != "other"]
)

,field,value,conf,evidence (verbatim)
0,investment_approach,fundamental,high,managing the only long short fundamental equity portfolio at Meridian Capital Pa...
1,market_side,buy_side,high,Primary investor responsible for managing the only long short fundamental equity...
2,sector,consumer,high,"Investment Analyst, Consumer & TMT – North53 Capital"
3,sector,technology,high,Principal analyst responsible for managing the Consumer and Technology portfolio...
4,sector,healthcare,high,Analyst – Healthcare Investment Banking
5,asset_class,equities,high,managing the only long short fundamental equity portfolio at Meridian Capital Pa...
6,team_leadership,Led 140+ members (student fraternity),medium,"Led 140+ members and organized signature events across Columbia Business School,..."
7,stated_metric · aum,$4.2bn gross portfolio,,Principal analyst responsible for managing the Consumer and Technology portfolio...


In [5]:
pd.DataFrame([{
    "firm": pos["firm"], "title": pos["title"],
    "dates": f'{pos["start_date"]} → {pos["end_date"] or "present"}',
    "type": pos["employment_type"],
    "investment role": pos["is_investment_role"],
} for pos in e["positions"]])

,firm,title,dates,type,investment role
0,Meridian Capital Partners,"Investment Professional, Generalist – Soft Catalyst & Fundamental Long/Short",2023-03 → present,professional,True
1,Millennium Management,"Investment Analyst, Consumer & TMT – North53 Capital",2021-09 → 2022-08,professional,True
2,Apollo Global Management,Associate – Private Equity,2019-08 → 2021-08,professional,True
3,J.P.Mogan,Analyst – Healthcare Investment Banking,2017-07 → 2019-07,professional,False
4,J.P.Mogan,Analyst – Healthcare Investment Banking (Summer Analyst),2016-06 → 2016-08,internship,False
5,Global Education Alliance,Co-Founder,2017-01 → present,volunteer,False
6,"Beta Alpha Psi, Gamma Chapter – Professional Business Fraternity","President, Alumni Relations Chair, Finance Committee",2014-02 → 2017-05,student_organization,False


In [6]:
# Everything the pipeline produces, per candidate: the model's extraction
# fields plus the enrichment computed on top. This is the full contract --
# nothing is parsed that is not listed here.
print("EXTRACTION (read by the model, with evidence)")
print("  " + ", ".join(sorted(e.keys())))
print()
print("ENRICHMENT (computed: knowledge base + arithmetic + checks)")
print("  " + ", ".join(sorted(k for k in ryan.keys() if k != "extraction")))

EXTRACTION (read by the model, with evidence)
  asset_classes, coverage, credentials, education, email, flags, full_name, investment_approach, languages, location_raw, market_side, methods, phone, positions, primary_sectors, software_tools, stated_metrics, team_leadership

ENRICHMENT (computed: knowledge base + arithmetic + checks)
  approach, approach_family, asset_classes, candidate_id, coverage_markets, coverage_markets_source, credentials_summary, current_firm, current_firm_type, display_name, employers, firm_types, firms, flags, has_buy_side_experience, has_sell_side_experience, investment_seniority_band, is_junior_range, languages, location, market_side, methods, name_source, non_professional_affiliations, platform_alum_of, quality, region, sectors, seniority_band, software_tools, source_file, years_experience, years_investment_experience


## 4 · Domain Knowledge and Entity Resolution

Some recruiting information should not be left to a language model to infer.

The `knowledge/` layer contains curated reference data for:

- Firm identities and firm / platform lineage
- Region and sector taxonomies
- Credential and terminology mappings
- Requisition requirements and scoring weights
- Human review rules for recurring data-quality observations

This allows the model to focus on resume interpretation while deterministic reference data handles facts that should be **consistent, explicit, and auditable**.

For example, a candidate may list a pod or investment platform rather than the parent firm. The resolver can connect that entity to the relevant platform without asking the model to guess.


In [7]:
from knowledge_base import KnowledgeBase, years_of_experience
kb = KnowledgeBase.load(ROOT / "knowledge")

# 1. Refusal to guess: four unrelated firms here begin with "Meridian".
#    A substring matcher would silently relocate a candidate to the wrong
#    continent; this one reports ambiguity instead of resolving.
print("resolve('Meridian')      ->", kb.resolve_firm("Meridian").method)

# 2. Pod-to-platform lineage: the resume names only the pod; the platform
#    exists only here. This is what lets the app surface "previously at
#    Millennium" for Ryan Patel.
print("lineage('North53 Capital') ->", kb.platform_lineage("North53 Capital"))

# 3. Tenure is date arithmetic, never model output. Overlapping positions
#    merge; internships and student societies are excluded -- counting them
#    added four years to one candidate in this pool.
overlap = [
    {"start_date": "2020-01", "end_date": "2022-01", "is_current": False,
     "employment_type": "professional", "is_investment_role": True},
    {"start_date": "2021-01", "end_date": "2023-01", "is_current": False,
     "employment_type": "professional", "is_investment_role": True},
]
print("overlapping 2y+2y roles  ->", years_of_experience(overlap), "years")

resolve('Meridian')      -> ambiguous
lineage('North53 Capital') -> ['Millennium Management']
overlapping 2y+2y roles  -> 3.0 years


### Human review is part of the data model

Not every flag should become a rejection.

The human triage layer records when an observed issue is benign — for example, formatting conventions or an internship embedded within an academic program. The original observation is retained, the review decision is recorded, and the change remains reversible.

This reflects an important product principle:

> **Automation should surface judgment calls, not hide them.**

The system can therefore distinguish between a genuine data-quality concern and an issue that a reviewer has already assessed as harmless.


## 5 · Candidate Matching: Eligibility → Ranking

A good candidate search answers two different questions:

1. **Is the candidate eligible for this role?**
2. **Among eligible candidates, who should I review first?**

The system keeps those decisions separate.

### Required criteria determine eligibility

Role requirements such as region, investment approach, sector, and experience band are treated as hard constraints **only when the requisition supports that interpretation**.

A hard constraint can therefore remove a candidate from the qualified pool.

### Ranking signals determine priority

Candidates who meet the required criteria are ranked using job-relevant signals such as:

- Sector fit
- Requirement-to-resume relevance
- Relevant skills
- Firm type
- Market coverage
- Credentials
- Platform / firm lineage
- Buy-side experience

The weights live outside the code in the requisition configuration, making the scoring logic explicit and reviewable.

### Near matches stay visible — but separate

A candidate who misses **exactly one** required criterion is shown as a **Near Match**, with the specific gap stated.

This is intentionally different from treating the candidate as qualified. It gives BD a controlled way to ask:

> **Is this requirement truly non-negotiable for this search?**

Candidates who fail multiple required criteria are not promoted into the shortlist simply to make the results look fuller.

### Different roles require different evidence

A junior analyst search should not be evaluated like a senior investor search. Early-career candidates may have limited track-record information, so education, relevant coursework, credentials, technical skills, internships, and sector / market exposure can carry more weight.

The scoring framework is therefore **role-specific rather than one-size-fits-all**.


In [8]:
from match import Requisitions, match_all
store = Requisitions.load(ROOT / "knowledge")

rows = []
for spec in store.items:
    exact, near = match_all(candidates, spec, store=store)
    rows.append({
        "requisition": spec["title"],
        "source": spec.get("source", ""),
        "qualify": len(exact),
        "one gap away": len(near),
        "top match": (f'{exact[0].display_name} ({exact[0].soft_score:.0%})'
                      if exact else "—"),
    })
pd.DataFrame(rows)

,requisition,source,qualify,one gap away,top match
0,Equity Analyst - US Healthcare Therapeutics,Point72 posting (real),2,6,Marcus Chen-Rodriguez (61%)
1,US Healthcare Origination Associate,Millennium posting (real),2,7,Ryan Patel (56%)
2,"Research Analyst, Healthcare (Mumbai)",Millennium posting (real),0,4,—
3,"Quantitative Analyst, Quantitative Strategies",Millennium posting (real),0,1,—


In [9]:
# The result the design is proudest of: an honest zero. Against the Mumbai
# posting's 4-5 year band, nobody qualifies -- and instead of a confidently
# ranked list, the system names the single gap for each near miss.
spec = store.get("mlm_mumbai_healthcare_research")
exact, near = match_all(candidates, spec, store=store)
pd.DataFrame([{
    "candidate": r.display_name,
    "fit (soft)": f"{r.soft_score:.0%}",
    "the one gap": f"{r.failed_hard[0].label}: has {r.failed_hard[0].found}, "
                   f"role needs {r.failed_hard[0].required}",
} for r in near])

,candidate,fit (soft),the one gap
0,Priya Nakamura,60%,"Investment experience: has 12.7 years, role needs 4-5 years"
1,Dr. Zara Al-Rashid,58%,"Investment experience: has 10.6 years, role needs 4-5 years"
2,Viktor Sharat,57%,"Investment experience: has 9.8 years, role needs 4-5 years"
3,MARINA SILVA COSTA,54%,"Region: has Europe, role needs APAC"


## 6 · Making the Scoring System Auditable

During development, requirement-similarity scores looked plausible but were systematically inflated.

The root cause became visible only after inspecting the **evidence quote** behind a score: a sentence about a "Biomodeller Trainee" had been treated as evidence for a fundamental India-equity requirement.

A concept-map term had been matched as a raw substring. Because the term was a single character, it appeared in almost every sentence.

The fix was deterministic:

- Match complete tokens rather than arbitrary substrings.
- Damp scores from very short sentences.
- Keep the evidence quote visible so a reviewer can inspect the basis of a score.

This led to a broader design rule:

> **If a scoring claim cannot show its evidence, it is difficult to audit.**

Evidence is therefore part of the data contract, not a decorative field.


## 7 · Evaluation Against Blind Human Screening

The goal of the evaluation is not to claim that a 40-pair experiment proves recruiter-level accuracy.

Instead, it asks a more useful product question:

> **Where does the system disagree with a human screener, and why?**

A reviewer independently labelled all **40 candidate-role pairs** without seeing the system's verdict. Labels were Y / N / borderline, with borderline cases excluded from the headline counts.

The system was evaluated on its **qualified / exact-match pool**; Near Matches were kept separate.


In [10]:
from evaluate import evaluate
ev = evaluate()
per = pd.DataFrame(ev["per_role"])[
    ["role", "precision", "recall", "agreement", "tp", "fp", "fn", "tn"]]
o = ev["overall"]
print(f'OVERALL  precision {o["precision"]:.0%}  recall {o["recall"]:.0%}  '
      f'agreement {o["agreement"]:.0%}  (n={o["judged"]} judged, '
      f'{len(ev["borderline"])} borderline set aside)')
per

OVERALL  precision 100%  recall 57%  agreement 92%  (n=36 judged, 4 borderline set aside)


,role,precision,recall,agreement,tp,fp,fn,tn
0,Equity Analyst - US Healthcare Therapeutics,1.0,1.0,1.0,2,0,0,8
1,US Healthcare Origination Associate,1.0,1.0,1.0,2,0,0,8
2,"Research Analyst, Healthcare (Mumbai)",NaN,0.0,0.7,0,0,3,7
3,"Quantitative Analyst, Quantitative Strategies",NaN,NaN,1.0,0,0,0,6


In [11]:
pd.DataFrame(ev["disagreements"])

,role,candidate,kind,system_reason
0,"Research Analyst, Healthcare (Mumbai)",Priya Nakamura,false_negative,"Investment experience: has 12.7 years, role needs 4-5 years"
1,"Research Analyst, Healthcare (Mumbai)",Viktor Sharat,false_negative,"Investment experience: has 9.8 years, role needs 4-5 years"
2,"Research Analyst, Healthcare (Mumbai)",Zara Al-Rashid,false_negative,"Investment experience: has 10.6 years, role needs 4-5 years"


### What the disagreements tell us

On this small test set, the system achieved **100% precision and 57% recall**, with **92% agreement** on the evaluated decisions.

The missed candidates had one common cause: the Mumbai role specified a 4–5 year experience band, while the human reviewer was willing to consider substantially more experienced candidates when the candidate pool was thin.

The important insight is not that the threshold should simply be softened.

It is that **recruiting criteria can be elastic, and that elasticity should be visible and controlled by the user rather than hidden inside the model.**

The product response is therefore:

- Keep the requisition's stated requirement intact.
- Surface one-gap candidates separately.
- Show the exact requirement and candidate value.
- Let the BD user decide whether to widen the search.

The same evaluation harness can be rerun after changes to prompts, thresholds, weights, or matching backends.

At this sample size, the percentages are directional rather than statistically conclusive. The value of the test is identifying the rule that needs product-level attention.


## 8 · Product Walkthrough

**Live application:** https://m-case-study-jasmine.streamlit.app/

The public deployment uses precomputed candidate data and does not expose a live model or API key.

The workflow is designed around the way a BD user actually screens talent.

### Step 1 · Define the search

The user can:

- Select a role from the job library
- Define custom criteria
- Browse the full talent pool without a requisition

The role determines which requirements are fixed; the remaining dimensions stay available for refinement.

![Choosing how to filter](https://raw.githubusercontent.com/yc4379-commits/m-case-study/main/docs/tour_modes.png)

### Step 2 · Refine the candidate pool

In role-based search, required dimensions are locked by the requisition. In browse mode, the user can explore market, investment approach, sector, market side, asset class, and keywords.

Advanced filters support the signals that can be useful in junior-analyst screening, including software, credentials, experience, and minimum parsing confidence.

![Advanced filters](https://raw.githubusercontent.com/yc4379-commits/m-case-study/main/docs/tour_advanced.png)

### Step 3 · Review qualified candidates and near matches separately

Qualified candidates are ranked within the eligible pool.

Near Matches are shown separately and explain the single requirement they miss. This makes the trade-off explicit instead of blending an ineligible candidate into the main ranking.

![The result list](https://raw.githubusercontent.com/yc4379-commits/m-case-study/main/docs/tour_results.png)

At higher volume, the same candidate data can be reviewed in a sortable table with CSV export.

![Table view](https://raw.githubusercontent.com/yc4379-commits/m-case-study/main/docs/app_table.png)

### Step 4 · Review the candidate

![Candidates view](https://raw.githubusercontent.com/yc4379-commits/m-case-study/main/docs/app_candidates.png)

The candidate page is organized around six screening questions:

- **Fit** — Does the match score hold up?
- **Profile** — What is the candidate's relevant background?
- **Figures** — What quantitative claims appear in the resume?
- **Issues** — What should be reviewed or verified?
- **Outreach** — What is a relevant opening for the candidate?
- **Full Record** — What exactly was extracted?

The candidate's match score is displayed together with eligibility and supporting signals. Parsing quality and review flags remain visible rather than being buried in an administrative screen.

![The candidate profile views](https://raw.githubusercontent.com/yc4379-commits/m-case-study/main/docs/tour_profile_tabs.png)

The **Outreach** view connects screening to the next BD action: drafting an initial message based on the candidate's relevant, verified experience.

The **Full Record** view provides transparency into the structured data behind the interface.


## 9 · Designing for Scale

The prototype is intentionally small, but each major component has a clear production successor.

| Component | Current prototype | Production direction |
|---|---|---|
| Parsing | Sequential API calls with content-hash caching | Queue + batch processing |
| Requirement relevance | Curated concept map | Pluggable semantic retrieval / embeddings |
| Search | In-memory filtering | Search index + semantic retrieval |
| Entity resolution | Curated firm and platform mappings | Governed entity graph / licensed reference data |
| Quality & evaluation | Record-level confidence + blind human evaluation | Sampled human review + drift monitoring |
| Serving | Streamlit public prototype | Internal deployment with managed credentials |

The important scaling principle is not simply infrastructure.

**The evaluation harness must scale with the system.** Any change to a prompt, threshold, scoring weight, or matching backend should be testable against a fixed set of human-labelled cases before it reaches production.


## 10 · Roadmap

The next stage is to extend the structured candidate layer into a broader talent intelligence workflow.

### 1. Internal sourcing knowledge

Connect candidate data with internal sourcing notes, meeting records, and call summaries so BD can answer questions that resumes alone cannot answer.

**Example:** “Who impressed the team on biotech last year?”

The same evidence and permission controls should apply to retrieved internal information.

### 2. In-app evaluation

Move the human-label workflow into the product so BD can label candidates, compare decisions, and monitor how matching performance changes over time.

### 3. Governed JD ingestion

Allow users to upload or paste a job description and convert it into the same structured requirement format used by the job library.

### 4. Talent Network

Extend the current firm / platform mappings into a queryable relationship layer connecting candidates, firms, platforms, and investment teams.

### 5. Verified performance data

If licensed external sources become available, quantitative claims such as AUM or track record can be validated rather than treated as self-reported resume information.

### Product vision

The long-term opportunity is not to add AI features for their own sake.

It is to build a **trusted talent data layer** that connects:

**Candidate data → Matching → Human review → Outreach → Pool insights → Internal sourcing knowledge**

AI can make each layer more useful, but the foundation remains the same: **reliable data, explicit logic, and evidence that a recruiter can review.**


## Appendix · Reproducing the Analysis

```bash
pip install -r requirements.txt
streamlit run app.py

pip install -r requirements-dev.txt
python -m pytest tests/ -q
python src/evaluate.py

# Full rebuild from raw resumes:
# requires resumes in data/resumes/ and ANTHROPIC_API_KEY in .env
python src/build_dataset.py
python tools/build_notebook.py
```

### Closing takeaway

This case study is intentionally not an “AI resume parser” demo.

It is a **talent intelligence workflow** where AI handles unstructured interpretation, deterministic systems enforce business rules, evidence makes decisions reviewable, and the interface turns structured data into practical BD actions.
